In [ ]:
import os
from pathlib import Path

import pandas as pd
from src import analysis, util
from src.analysis import evaluation

os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Use CPU for evaluation

In [ ]:
# ======================================================================
# Configuration
# ======================================================================
# Models / architectures to compare
MODELS = [
    # --- FNO ---
    # FNO small
    "FNO_m48x48_h64_l4_lhs_var80_seed3001_20260110_085608",
    # FNO big
    # --- PI-FNO ---
    # PI-FNO small
    "PI-FNO_m12x12_h24_l4_lamPhys1e-04_lamP5e-03_lhs_var80_seed3001_20260109_191704",
    # PI-FNO big
    # --- U-NO ---
    # U-NO small
    # U-NO big
    # --- PI-U-NO ---
    # PI-U-NO small
    # PI-U-NO big
]

MODEL_LABELS = {
    # "FNO_lhs_var80_plog100_seed3001_20251212_125538": "FNO_m32x32_h64_l4",
    # "FNO_lhs_var80_plog100_seed3001_20260103_121739": "FNO_m12x12_h24_l4",
}

# Shared datasets (IDENTICAL for all models)
ID_DATASET = "lhs_var80_seed3001"
OOD_DATASET = "lhs_var120_seed4001"

show_id_evaluation = True
show_ood_evaluation = False

In [ ]:
# ======================================================================
# Paths
# ======================================================================
RAW_ROOT = Path("../../data/raw")
PROCESSED_ROOT = Path("../data/processed")


def run_or_load_artifacts_evaluation(
    *,
    model_name: str,
    dataset_name: str,
) -> pd.DataFrame:
    """Load or generate evaluation artifacts for one model on one dataset."""
    run_dir = PROCESSED_ROOT / model_name
    checkpoint_path = run_dir / "best_model_state_dict.pt"

    dataset_path = RAW_ROOT / dataset_name / "cases"

    save_root = run_dir / "analysis" / "id" if dataset_name == ID_DATASET else run_dir / "analysis" / "ood" / dataset_name

    parquet_path = save_root / f"{dataset_name}.parquet"

    if parquet_path.exists():
        print(f"[INFO] Loading {model_name} | {dataset_name}")
        return pd.read_parquet(parquet_path)

    print(f"[INFO] Creating artifacts: {model_name} | {dataset_name}")

    model, loader, processor, device = analysis.analysis_interference.load_inference_context(
        dataset_path=dataset_path,
        checkpoint_path=checkpoint_path,
        batch_size=1,
    )

    df, _ = analysis.analysis_artifacts.generate_artifacts(
        model=model,
        loader=loader,
        processor=processor,
        device=device,
        save_root=save_root,
        dataset_name=dataset_name,
    )

    return df

In [ ]:
# ======================================================================
# Load evaluation data
# ======================================================================
datasets_eval_id = {}
datasets_eval_ood = {}

for model_name in MODELS:
    print(f"\n=== {model_name} ===")

    # -----------------
    # ID
    # -----------------
    df_raw_id = run_or_load_artifacts_evaluation(
        model_name=model_name,
        dataset_name=ID_DATASET,
    )
    df_id = evaluation.evaluation_dataframe.build_eval_df(df_raw_id)
    label = MODEL_LABELS.get(model_name, model_name)
    datasets_eval_id[label] = df_id

    # -----------------
    # OOD
    # -----------------
    df_raw_ood = run_or_load_artifacts_evaluation(
        model_name=model_name,
        dataset_name=OOD_DATASET,
    )
    df_ood = evaluation.evaluation_dataframe.build_eval_df(df_raw_ood)
    label = MODEL_LABELS.get(model_name, model_name)
    datasets_eval_ood[label] = df_ood

In [ ]:
print(df_id.columns.tolist())

In [ ]:
import numpy as np

npz_path = df_raw_id.loc[0, "npz_path"]  # oder beliebiger Index
data = np.load(npz_path, allow_pickle=True)

print("Keys im NPZ:")
for k in data.files:
    print(" -", k)

for k in data.files:
    arr = data[k]
    print(f"{k}: type={type(arr)}, dtype={arr.dtype}, shape={getattr(arr, 'shape', None)}")


In [ ]:
if show_id_evaluation:
    # =====================================================================
    # TOGGLE SETUP (MODELS)
    # =====================================================================
    toggle = util.util_nb.make_toggle_shortcut(dfs=datasets_eval_id)

    # =====================================================================
    # 0. OVERVIEW
    # =====================================================================
    overview_plots = [
        toggle("Overview: Global comparison summary", evaluation.evaluation_plot.evaluation_plot_overview_scoreboard.plot_overview_scoreboard),
        toggle(
            "Overview: Pareto (Error vs Physics)",
            evaluation.evaluation_plot.evaluation_plot_overview_scoreboard.plot_overview_pareto_error_vs_physics,
        ),
    ]

    # =====================================================================
    # 1. GLOBAL ERROR ANALYSIS
    # =====================================================================
    global_error_analysis_plots = [
        toggle("1-1. Global error metrics", evaluation.evaluation_plot.evaluation_plot_global_error_analysis.plot_global_error_metrics),
        toggle("1-2. Global error distribution", evaluation.evaluation_plot.evaluation_plot_global_error_analysis.plot_error_distribution),
        toggle("1-3. GT vs Prediction (mean)", evaluation.evaluation_plot.evaluation_plot_global_error_analysis.plot_global_gt_vs_pred),
        toggle("1-4. Mean error maps", evaluation.evaluation_plot.evaluation_plot_global_error_analysis.plot_mean_error_maps),
        toggle("1-5. Std error maps", evaluation.evaluation_plot.evaluation_plot_global_error_analysis.plot_std_error_maps),
        toggle(
            "1-6. Error frequency spectrum", evaluation.evaluation_plot.evaluation_plot_global_error_analysis.plot_global_error_frequency_spectrum
        ),
    ]

    # =====================================================================
    # 2. ARCHITECTURE SENSITIVITY
    # =====================================================================
    architecture_sensitivity_plots = [
        toggle(
            "2-1. Error vs architecture parameters",
            evaluation.evaluation_plot.evaluation_plot_architecture_sensitivity.plot_error_vs_architecture_parameters,
        ),
        toggle("2-2. Capacity vs performance", evaluation.evaluation_plot.evaluation_plot_architecture_sensitivity.plot_capacity_vs_performance),
        toggle("2-3. Parameter efficiency", evaluation.evaluation_plot.evaluation_plot_architecture_sensitivity.plot_parameter_efficiency),
    ]

    # =====================================================================
    # 3. ERROR DECOMPOSITION
    # =====================================================================
    error_decomposition_plots = [
        toggle("3-1. Error vs |GT| magnitude", evaluation.evaluation_plot.evaluation_plot_error_decomposition.plot_error_vs_gt_magnitude),
        toggle("3-2. Boundary vs interior error", evaluation.evaluation_plot.evaluation_plot_error_decomposition.plot_error_vs_boundary_distance),
    ]

    # =====================================================================
    # 4. PHYSICAL CONSISTENCY
    # =====================================================================
    physical_consistency_plots = [
        toggle("4-1. Velocity divergence (∇·u)", evaluation.evaluation_plot.evaluation_plot_physical_consistency.plot_velocity_divergence),
        toggle("4-2. Mass conservation error map", evaluation.evaluation_plot.evaluation_plot_physical_consistency.plot_mass_conservation_error_map),
        toggle(
            "4-3. Pressure boundary consistency (p_bc)", evaluation.evaluation_plot.evaluation_plot_physical_consistency.plot_pressure_bc_consistency
        ),
        toggle("4-4. Darcy-Brinkman operator residual", evaluation.evaluation_plot.evaluation_plot_physical_consistency.plot_brinkman_residual),
    ]

    # =====================================================================
    # 5. SPECTRAL & REPRESENTATION ANALYSIS
    # =====================================================================
    spectral_analysis_plots = [
        toggle(
            "5-1. Learned spectral energy per layer",
            evaluation.evaluation_plot.evaluation_plot_spectral_analysis.plot_learned_spectral_energy_per_layer,
        ),
        toggle("5-2. Spectral centroid vs depth", evaluation.evaluation_plot.evaluation_plot_spectral_analysis.plot_spectral_centroid_vs_depth),
        toggle(
            "5-3. Learned spectrum vs error spectrum",
            evaluation.evaluation_plot.evaluation_plot_spectral_analysis.plot_learned_spectrum_vs_error_spectrum,
        ),
    ]

    # =====================================================================
    # 6. PARAMETER SENSITIVITY
    # =====================================================================
    error_sensitivity_plots = [
        toggle(
            "6-1. Parameter-error correlation (heatmap)",
            evaluation.evaluation_plot.evaluation_plot_parameter_sensitivity.plot_parameter_error_heatmap,
        ),
        toggle(
            "6-2. Error vs input parameter (binned trend)",
            evaluation.evaluation_plot.evaluation_plot_parameter_sensitivity.plot_error_vs_parameter_trend,
        ),
    ]

    # =====================================================================
    # FINAL PANEL
    # =====================================================================
    sections = [
        util.util_nb.make_dropdown_section(overview_plots),
        util.util_nb.make_dropdown_section(global_error_analysis_plots),
        util.util_nb.make_dropdown_section(architecture_sensitivity_plots),
        util.util_nb.make_dropdown_section(error_decomposition_plots),
        util.util_nb.make_dropdown_section(physical_consistency_plots),
        util.util_nb.make_dropdown_section(spectral_analysis_plots),
        util.util_nb.make_dropdown_section(error_sensitivity_plots),
    ]

    tab_titles = [
        "Overview",
        "1. Global Error Analysis",
        "2. Architecture Sensitivity",
        "3. Error Decomposition",
        "4. Physical Consistency",
        "5. Spectral & Representation Analysis",
        "6. Error Sensitivity (Input Parameters)",
    ]

    evaluation_panel = util.util_nb.make_lazy_panel_with_tabs(
        sections,
        tab_titles=tab_titles,
        open_btn_text="Model Comparison OOD - Open Evaluation",
        close_btn_text="Close",
    )

    display(evaluation_panel)

In [ ]:
if show_ood_evaluation:
    # =====================================================================
    # TOGGLE SETUP (MODELS)
    # =====================================================================
    toggle = util.util_nb.make_toggle_shortcut(dfs=datasets_eval_id)

    # =====================================================================
    # 0. OVERVIEW
    # =====================================================================
    overview_plots = [
        toggle("Overview: Global comparison summary", evaluation.evaluation_plot.evaluation_plot_overview_scoreboard.plot_overview_scoreboard),
        toggle(
            "Overview: Pareto (Error vs Physics)",
            evaluation.evaluation_plot.evaluation_plot_overview_scoreboard.plot_overview_pareto_error_vs_physics,
        ),
    ]

    # =====================================================================
    # 1. GLOBAL ERROR ANALYSIS
    # =====================================================================
    global_error_analysis_plots = [
        toggle("1-1. Global error metrics", evaluation.evaluation_plot.evaluation_plot_global_error_analysis.plot_global_error_metrics),
        toggle("1-2. Global error distribution", evaluation.evaluation_plot.evaluation_plot_global_error_analysis.plot_error_distribution),
        toggle("1-3. GT vs Prediction (mean)", evaluation.evaluation_plot.evaluation_plot_global_error_analysis.plot_global_gt_vs_pred),
        toggle("1-4. Mean error maps", evaluation.evaluation_plot.evaluation_plot_global_error_analysis.plot_mean_error_maps),
        toggle("1-5. Std error maps", evaluation.evaluation_plot.evaluation_plot_global_error_analysis.plot_std_error_maps),
        toggle(
            "1-6. Error frequency spectrum", evaluation.evaluation_plot.evaluation_plot_global_error_analysis.plot_global_error_frequency_spectrum
        ),
    ]

    # =====================================================================
    # 2. ARCHITECTURE SENSITIVITY
    # =====================================================================
    architecture_sensitivity_plots = [
        toggle(
            "2-1. Error vs architecture parameters",
            evaluation.evaluation_plot.evaluation_plot_architecture_sensitivity.plot_error_vs_architecture_parameters,
        ),
        toggle("2-2. Capacity vs performance", evaluation.evaluation_plot.evaluation_plot_architecture_sensitivity.plot_capacity_vs_performance),
        toggle("2-3. Parameter efficiency", evaluation.evaluation_plot.evaluation_plot_architecture_sensitivity.plot_parameter_efficiency),
    ]

    # =====================================================================
    # 3. ERROR DECOMPOSITION
    # =====================================================================
    error_decomposition_plots = [
        toggle("3-1. Error vs |GT| magnitude", evaluation.evaluation_plot.evaluation_plot_error_decomposition.plot_error_vs_gt_magnitude),
        toggle("3-2. Boundary vs interior error", evaluation.evaluation_plot.evaluation_plot_error_decomposition.plot_error_vs_boundary_distance),
    ]

    # =====================================================================
    # 4. PHYSICAL CONSISTENCY
    # =====================================================================
    physical_consistency_plots = [
        toggle("4-1. Velocity divergence (∇·u)", evaluation.evaluation_plot.evaluation_plot_physical_consistency.plot_velocity_divergence),
        toggle("4-2. Mass conservation error map", evaluation.evaluation_plot.evaluation_plot_physical_consistency.plot_mass_conservation_error_map),
        toggle(
            "4-3. Pressure boundary consistency (p_bc)", evaluation.evaluation_plot.evaluation_plot_physical_consistency.plot_pressure_bc_consistency
        ),
        toggle("4-4. Darcy-Brinkman operator residual", evaluation.evaluation_plot.evaluation_plot_physical_consistency.plot_brinkman_residual),
    ]

    # =====================================================================
    # 5. SPECTRAL & REPRESENTATION ANALYSIS
    # =====================================================================
    spectral_analysis_plots = [
        toggle(
            "5-1. Learned spectral energy per layer",
            evaluation.evaluation_plot.evaluation_plot_spectral_analysis.plot_learned_spectral_energy_per_layer,
        ),
        toggle("5-2. Spectral centroid vs depth", evaluation.evaluation_plot.evaluation_plot_spectral_analysis.plot_spectral_centroid_vs_depth),
        toggle(
            "5-3. Learned spectrum vs error spectrum",
            evaluation.evaluation_plot.evaluation_plot_spectral_analysis.plot_learned_spectrum_vs_error_spectrum,
        ),
    ]

    # =====================================================================
    # 6. PARAMETER SENSITIVITY
    # =====================================================================
    error_sensitivity_plots = [
        toggle(
            "6-1. Parameter-error correlation (heatmap)",
            evaluation.evaluation_plot.evaluation_plot_parameter_sensitivity.plot_parameter_error_heatmap,
        ),
        toggle(
            "6-2. Error vs input parameter (binned trend)",
            evaluation.evaluation_plot.evaluation_plot_parameter_sensitivity.plot_error_vs_parameter_trend,
        ),
    ]

    # =====================================================================
    # FINAL PANEL
    # =====================================================================
    sections = [
        util.util_nb.make_dropdown_section(overview_plots),
        util.util_nb.make_dropdown_section(global_error_analysis_plots),
        util.util_nb.make_dropdown_section(architecture_sensitivity_plots),
        util.util_nb.make_dropdown_section(error_decomposition_plots),
        util.util_nb.make_dropdown_section(physical_consistency_plots),
        util.util_nb.make_dropdown_section(spectral_analysis_plots),
        util.util_nb.make_dropdown_section(error_sensitivity_plots),
    ]

    tab_titles = [
        "Overview",
        "1. Global Error Analysis",
        "2. Architecture Sensitivity",
        "3. Error Decomposition",
        "4. Physical Consistency",
        "5. Spectral & Representation Analysis",
        "6. Error Sensitivity (Input Parameters)",
    ]

    evaluation_panel = util.util_nb.make_lazy_panel_with_tabs(
        sections,
        tab_titles=tab_titles,
        open_btn_text="Model Comparison ID - Open Evaluation",
        close_btn_text="Close",
    )

    display(evaluation_panel)